In [ ]:
#Chapter 2 Exercise solutions
# 第二章 习题参考答案

In [ ]:
# 从 importlib.metadata 中导入 version 函数,用于查询已安装第三方库的版本号
from importlib.metadata import version

# 打印当前环境中 torch(PyTorch)的版本号
print("torch version:", version("torch"))
# 打印当前环境中 tiktoken(OpenAI 提供的 BPE 分词器库)的版本号
print("tiktoken version:", version("tiktoken"))

In [ ]:
# 导入 tiktoken 分词库
import tiktoken

# 加载 GPT-2 所使用的 BPE(字节对编码, Byte Pair Encoding)分词器
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# 习题 2.1:对包含生僻/未登录组合的字符串 "Akwirw ier" 进行编码,
# 观察 BPE 分词器会将其拆分成哪些子词(subword) token id
integers = tokenizer.encode("Akwirw ier")
print(integers)

In [ ]:
# 逐个把上一步得到的 token id 解码还原成文本片段,
# 直观查看每个 token 具体对应哪一段子词
for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")

In [ ]:
# 单独编码子串 "Ak",验证其 token id 是否与上面拆分出的第一个 token 一致
tokenizer.encode("Ak")

In [ ]:
# 单独编码单个字符 "w",验证其 token id 是否与上面对应位置的 token 一致
tokenizer.encode("w")

In [ ]:
# 单独编码子串 "ir",验证其 token id 是否与上面对应位置的 token 一致
tokenizer.encode("ir")

In [ ]:
# 再次单独编码字符 "w"(对应 "Akwirw" 中第二次出现的 "w"),
# 验证其 token id 与之前 "w" 的编码结果相同(同一子词复用同一 token id)
tokenizer.encode("w")

In [ ]:
# 单独编码空格 " ",验证空格在 BPE 词表中也会被编码为独立的 token
tokenizer.encode(" ")

In [ ]:
# 单独编码子串 "ier",验证其 token id 是否与上面对应位置的 token 一致
tokenizer.encode("ier")

In [ ]:
# 将前面各步骤依次得到的 token id 拼接起来解码,
# 验证能否完整还原出原始字符串 "Akwirw ier",
# 从而证明 BPE 分词/解码过程是可逆的(编码再解码不丢失信息)
tokenizer.decode([33901, 86, 343, 86, 220, 959])

In [ ]:
# Exercise 2.2
# 习题 2.2:实现基于滑动窗口(sliding window)的数据集与数据加载器(DataLoader)

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


# 自定义 Dataset:把整段原始文本编码成 token id 序列后,
# 用滑动窗口切分成多个 (输入, 目标) 样本对,用于下一个 token 预测任务的训练
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        # 对整段文本进行分词编码(允许出现特殊 token "<|endoftext|>")
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        # 用滑动窗口把长 token 序列切分成若干个长度为 max_length 的(可能有重叠的)片段:
        # - 输入片段 input_chunk 为 [i, i+max_length)
        # - 目标片段 target_chunk 是输入整体右移一位,即 [i+1, i+max_length+1),
        #   代表"预测下一个 token"这一自回归训练目标
        # stride 控制窗口每次滑动的步长,stride < max_length 时片段之间会有重叠
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # 返回数据集中样本(切分出的片段)的总数
        return len(self.input_ids)

    def __getitem__(self, idx):
        # 根据索引返回一对 (输入张量, 目标张量)
        return self.input_ids[idx], self.target_ids[idx]


# 封装函数:给定原始文本,构建对应的 DataLoader,便于按批次(batch)迭代训练数据
def create_dataloader(txt, batch_size=4, max_length=256, stride=128):
    # Initialize the tokenizer
    # 初始化 GPT-2 的 BPE 分词器
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    # 创建上面定义的滑动窗口数据集
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    # 创建 PyTorch DataLoader,按 batch_size 打包样本
    # (注意:此处未设置 shuffle/drop_last,为该练习的简化版本)
    dataloader = DataLoader(dataset, batch_size=batch_size)

    return dataloader


# 读取本地文本文件 the-verdict.txt 作为训练语料
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 重新获取一次分词器,并对全文进行编码,得到完整的 token id 序列
tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

In [ ]:
# 构建一个 DataLoader:batch_size=4,输入序列长度 max_length=2,滑动步长 stride=2
# 由于 stride == max_length,相邻样本之间没有重叠(窗口不重叠地依次滑动)
dataloader = create_dataloader(raw_text, batch_size=4, max_length=2, stride=2)

# 取出第一个 batch 后立即跳出循环,只查看第一批数据
for batch in dataloader:
    x, y = batch
    break

# 打印这一 batch 的输入张量 x,形状应为 (batch_size=4, max_length=2)
x

In [ ]:
# 构建另一个 DataLoader:batch_size=4,输入序列长度 max_length=8,滑动步长 stride=2
# 由于 stride(2) < max_length(8),相邻样本之间会有重叠(overlap),
# 对比上一单元可以直观看到 max_length 增大后,输入张量形状(每行长度)的变化
dataloader = create_dataloader(raw_text, batch_size=4, max_length=8, stride=2)

for batch in dataloader:
    x, y = batch
    break

# 打印这一 batch 的输入张量 x,形状应为 (batch_size=4, max_length=8)
x